# 02 - Clean monthly panel

This notebook converts the raw long-format Banco Central do Brasil SGS data downloaded in `01_download_bcb_sgs.ipynb` into a clean monthly wide panel.

The output panel has one row per month and one column per series. It will be the shared input for the descriptive plots and local projection estimates later in the project.

## Imports and paths

We load the core Python packages used throughout the project and import the transformation helpers from `src/transforms.py`. The path block is written so the notebook can run from either the project root or the `notebooks/` directory.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.transforms import (
    add_credit_shares,
    add_growth_rates,
    add_log_credit_variables,
    add_policy_variables,
    add_state_variables,
    ensure_datetime,
    first_last_nonmissing,
    missing_summary,
    monthly_panel_from_long,
)

RAW_FILE = PROJECT_ROOT / "data" / "raw" / "bcb_sgs_all_long.csv"
DICTIONARY_FILE = PROJECT_ROOT / "data" / "series_dictionary.csv"
OUTPUT_FILE = PROJECT_ROOT / "data" / "processed" / "brazil_credit_monthly_panel.csv"

In [2]:

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [3]:

from src.transforms import (
    add_credit_shares,
    add_growth_rates,
    add_log_credit_variables,
    add_policy_variables,
    add_state_variables,
    ensure_datetime,
    first_last_nonmissing,
    missing_summary,
    monthly_panel_from_long,
)


In [4]:

RAW_FILE = PROJECT_ROOT / "data" / "raw" / "bcb_sgs_all_long.csv"
DICTIONARY_FILE = PROJECT_ROOT / "data" / "series_dictionary.csv"
OUTPUT_FILE = PROJECT_ROOT / "data" / "processed" / "brazil_credit_monthly_panel.csv"

## Load raw data and series dictionary

The raw file is a tidy, long-format table with one observation per date-series pair. The series dictionary gives the manually verified SGS code, frequency, and project name for each series, which we use below when aggregating daily series to months.

In [5]:
raw = pd.read_csv(RAW_FILE)
series_dictionary = pd.read_csv(DICTIONARY_FILE)

display(raw.head())
display(series_dictionary.head())

print(f"Raw observations: {len(raw):,}")
print(f"Raw series in file: {raw['series'].nunique():,}")
print(f"Series in dictionary: {len(series_dictionary):,}")

,date,value,series
0,2007-01-01,13.13,selic_monthly_annualized
1,2007-02-01,12.93,selic_monthly_annualized
2,2007-03-01,12.74,selic_monthly_annualized
3,2007-04-01,12.58,selic_monthly_annualized
4,2007-05-01,12.43,selic_monthly_annualized


,name,series_id,source,frequency,unit,notes,verified
0,selic_target,432,BCB_SGS,monthly,percent_annual,Selic target rate; main monetary policy rate,no
1,selic_daily,11,BCB_SGS,daily,percent_daily,Interest rate - Selic; daily series; aggregate...,no
2,selic_annual_daily,1178,BCB_SGS,daily,percent_annual,Interest rate - Selic in annual terms (basis 2...,no
3,selic_monthly_annualized,4189,BCB_SGS,monthly,percent_annual,Interest rate - Selic accumulated in the month...,yes
4,inflation_target,13521,BCB_SGS,annual,percent_annual,Inflation target; useful for policy-rule or in...,yes


Raw observations: 7,722
Raw series in file: 33
Series in dictionary: 40


## Basic raw-data checks

Before reshaping, we verify that the expected long-format columns are present, parse the date column, and summarize coverage by series. These checks help catch failed downloads, malformed dates, or unexpected missing values before they become harder to diagnose in wide format.

In [6]:
required_columns = {"date", "value", "series"}
missing_columns = required_columns.difference(raw.columns)
if missing_columns:
    raise ValueError(f"Raw data is missing required columns: {sorted(missing_columns)}")

raw = ensure_datetime(raw, date_col="date")

print("Missing values by raw column:")
display(raw.isna().sum().rename("missing_count").to_frame())

date_range_by_series = (
    raw.groupby("series")
    .agg(first_date=("date", "min"), last_date=("date", "max"))
    .sort_index()
)
display(date_range_by_series)

observations_by_series = (
    raw.groupby("series")
    .size()
    .rename("observations")
    .sort_values(ascending=False)
    .to_frame()
)
display(observations_by_series)

Missing values by raw column:


,missing_count
date,0
value,991
series,0


,first_date,last_date
series,,
credit_total_stock,2007-01-01,2026-06-01
credit_total_stock_firms,2007-01-01,2026-06-01
credit_total_stock_households,2007-01-01,2026-06-01
credit_total_stock_to_gdp,2007-01-01,2026-06-01
directed_credit_stock,2007-01-01,2026-06-01
directed_credit_stock_firms,2007-01-01,2026-06-01
directed_credit_stock_households,2007-01-01,2026-06-01
directed_credit_stock_to_gdp,2007-01-01,2026-06-01
exchange_rate_usd_sale_avg,2007-01-01,2026-06-01


,observations
series,
credit_total_stock,234
credit_total_stock_firms,234
credit_total_stock_households,234
credit_total_stock_to_gdp,234
directed_credit_stock,234
directed_credit_stock_firms,234
directed_credit_stock_households,234
directed_credit_stock_to_gdp,234
exchange_rate_usd_sale_avg,234


## Convert to monthly panel

The empirical analysis is monthly, so every series needs to be aligned to a common monthly index. Daily series are converted to monthly averages, while monthly series keep the last observed value within each month. The result is a wide panel with one row per month and one column per project series name.

In [7]:
panel = monthly_panel_from_long(raw, dictionary_df=series_dictionary)

display(panel.head())
print(f"Monthly panel shape: {panel.shape[0]:,} rows x {panel.shape[1]:,} columns")

,month,credit_total_stock,credit_total_stock_firms,credit_total_stock_households,credit_total_stock_to_gdp,directed_credit_stock,directed_credit_stock_firms,directed_credit_stock_households,directed_credit_stock_to_gdp,exchange_rate_usd_sale_avg,...,interest_rate_free_new_operations_firms,interest_rate_free_new_operations_households,interest_rate_free_new_operations_total,interest_rate_new_operations_firms,interest_rate_new_operations_households,interest_rate_new_operations_total,ipca,ipca_12m,selic_monthly_annualized,unemployment_rate_pnadc
0,2007-01-01,738456.0,NaN,NaN,30.33,NaN,NaN,NaN,NaN,2.1385,...,NaN,NaN,NaN,NaN,NaN,NaN,0.44,2.99,13.13,NaN
1,2007-02-01,748518.0,NaN,NaN,30.44,NaN,NaN,NaN,NaN,2.0963,...,NaN,NaN,NaN,NaN,NaN,NaN,0.44,3.02,12.93,NaN
2,2007-03-01,762353.0,413952.0,348401.0,30.66,271893.0,169316.0,102577.0,10.93,2.0887,...,NaN,NaN,NaN,NaN,NaN,NaN,0.37,2.96,12.74,NaN
3,2007-04-01,781888.0,424836.0,357052.0,31.06,276135.0,172021.0,104113.0,10.97,2.0320,...,NaN,NaN,NaN,NaN,NaN,NaN,0.25,3.00,12.58,NaN
4,2007-05-01,793133.0,425838.0,367295.0,31.16,278486.0,172644.0,105844.0,10.94,1.9816,...,NaN,NaN,NaN,NaN,NaN,NaN,0.28,3.18,12.43,NaN


Monthly panel shape: 234 rows x 34 columns


## Construct key variables

This step creates the credit shares, accounting check variables, log credit stocks, monthly credit growth rates, policy-rate changes, optional exchange-rate changes, and the high-directed-share regime indicator used in the state-dependent specifications. The credit-stock identities are project assumptions, so they are built explicitly rather than hidden inside later estimation code.

In [8]:
panel = add_credit_shares(panel)
panel = add_log_credit_variables(panel)
panel = add_growth_rates(panel)
panel = add_policy_variables(panel)
panel = add_state_variables(panel)

constructed_columns = [
    "directed_credit_share",
    "free_credit_share",
    "credit_gap_check",
    "credit_gap_check_pct",
    "log_credit_total_stock",
    "log_free_credit_stock",
    "log_directed_credit_stock",
    "growth_credit_total_stock",
    "growth_free_credit_stock",
    "growth_directed_credit_stock",
    "delta_selic",
    "exchange_rate_log_change",
    "high_directed_share",
]
display(panel[[col for col in constructed_columns if col in panel.columns]].head())

,directed_credit_share,free_credit_share,credit_gap_check,credit_gap_check_pct,log_credit_total_stock,log_free_credit_stock,log_directed_credit_stock,growth_credit_total_stock,growth_free_credit_stock,growth_directed_credit_stock,delta_selic,exchange_rate_log_change,high_directed_share
0,NaN,NaN,NaN,NaN,13.512317,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,13.525851,NaN,NaN,1.353373,NaN,NaN,-0.20,-1.993076,NaN
2,0.356650,0.643350,0.0,0.0,13.544165,13.103099,12.513164,1.831445,NaN,NaN,-0.19,-0.363202,0.0
3,0.353164,0.646836,0.0,0.0,13.569467,13.133804,12.528645,2.530180,3.070468,1.548127,-0.16,-2.752133,0.0
4,0.351121,0.648879,0.0,0.0,13.583746,13.151237,12.537123,1.427942,1.743282,0.847791,-0.15,-2.511593,0.0


## Diagnostics

These diagnostics are used to catch coding or concept mistakes before the cleaned panel is used in figures or local projections. In particular, the credit-gap check should be small if the total, free, and directed credit stock concepts line up as expected.

In [9]:
print(f"Panel date range: {panel['month'].min().date()} to {panel['month'].max().date()}")
print(f"Rows: {panel.shape[0]:,}")
print(f"Columns: {panel.shape[1]:,}")

print("Missing values by column:")
display(missing_summary(panel))

print("First and last non-missing date by column:")
display(first_last_nonmissing(panel, date_col="month"))

key_constructed_variables = [
    "directed_credit_share",
    "free_credit_share",
    "credit_gap_check_pct",
    "growth_credit_total_stock",
    "growth_free_credit_stock",
    "growth_directed_credit_stock",
    "delta_selic",
    "exchange_rate_log_change",
    "high_directed_share",
]
key_constructed_variables = [
    col for col in key_constructed_variables if col in panel.columns
]

print("Summary statistics for key constructed variables:")
display(panel[key_constructed_variables].describe().T)

print(
    "directed_credit_share min/max:",
    panel["directed_credit_share"].min(),
    panel["directed_credit_share"].max(),
)
print(
    "free_credit_share min/max:",
    panel["free_credit_share"].min(),
    panel["free_credit_share"].max(),
)
print(
    "credit_gap_check_pct mean absolute value:",
    panel["credit_gap_check_pct"].abs().mean(),
)
print(
    "credit_gap_check_pct max absolute value:",
    panel["credit_gap_check_pct"].abs().max(),
)

panel = panel.dropna(subset=["free_credit_stock", "directed_credit_stock"])

Panel date range: 2007-01-01 to 2026-06-01
Rows: 234
Columns: 50
Missing values by column:


,column,missing_count,missing_pct
0,month,0,0.000000
1,credit_total_stock,2,0.854701
2,credit_total_stock_firms,4,1.709402
3,credit_total_stock_households,4,1.709402
4,credit_total_stock_to_gdp,2,0.854701
5,directed_credit_stock,4,1.709402
6,directed_credit_stock_firms,4,1.709402
7,directed_credit_stock_households,4,1.709402
8,directed_credit_stock_to_gdp,4,1.709402
9,exchange_rate_usd_sale_avg,2,0.854701


First and last non-missing date by column:


,column,first_nonmissing,last_nonmissing
0,credit_total_stock,2007-01-01,2026-04-01
1,credit_total_stock_firms,2007-03-01,2026-04-01
2,credit_total_stock_households,2007-03-01,2026-04-01
3,credit_total_stock_to_gdp,2007-01-01,2026-04-01
4,directed_credit_stock,2007-03-01,2026-04-01
5,directed_credit_stock_firms,2007-03-01,2026-04-01
6,directed_credit_stock_households,2007-03-01,2026-04-01
7,directed_credit_stock_to_gdp,2007-03-01,2026-04-01
8,exchange_rate_usd_sale_avg,2007-01-01,2026-04-01
9,free_credit_stock,2007-03-01,2026-04-01


Summary statistics for key constructed variables:


,count,mean,std,min,25%,50%,75%,max
directed_credit_share,230.0,4.198565e-01,5.014996e-02,0.315663,0.394438,0.418289,0.458676,0.503853
free_credit_share,230.0,5.801435e-01,5.014992e-02,0.496147,0.541324,0.581711,0.605562,0.684337
credit_gap_check_pct,230.0,6.860969e-09,3.294258e-07,-0.000001,0.000000,0.000000,0.000000,0.000002
growth_credit_total_stock,231.0,9.885463e-01,8.620697e-01,-1.021717,0.345454,0.980372,1.533567,3.616832
growth_free_credit_stock,229.0,9.280918e-01,1.024789e+00,-1.523022,0.233189,0.908833,1.537757,4.303329
growth_directed_credit_stock,229.0,1.068000e+00,1.053653e+00,-0.930099,0.327524,0.942824,1.658763,7.299566
delta_selic,233.0,5.450644e-03,3.644231e-01,-1.000000,-0.190000,0.000000,0.190000,1.350000
exchange_rate_log_change,231.0,3.705331e-01,3.662037e+00,-9.100995,-2.075534,0.191084,2.380124,18.849826
high_directed_share,230.0,5.000000e-01,5.010905e-01,0.000000,0.000000,0.500000,1.000000,1.000000


directed_credit_share min/max: 0.31566305127911315 0.5038526924331652
free_credit_share min/max: 0.4961473075668348 0.6843369487208869
credit_gap_check_pct mean absolute value: 1.4373001719996056e-07
credit_gap_check_pct max absolute value: 1.6191133087875756e-06


Added a line at the end of the cell above to trim the panel to the last observation period where we have the main variables directed and not directed credit. 

## Save cleaned panel

Finally, we save the cleaned monthly panel to `data/processed/`. This CSV is intentionally generated output: it can be recreated by rerunning the download and cleaning notebooks.

In [10]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
panel.to_csv(OUTPUT_FILE, index=False)

print(f"Saved cleaned monthly panel to: {OUTPUT_FILE}")

Saved cleaned monthly panel to: c:\Users\chico\brazil-directed-credit-monetary-policy\data\processed\brazil_credit_monthly_panel.csv
